# 03 — Data Cleaning (missing, duplicates, types, strings, outliers)

Maps to **MODULE-03a / 03b**. Every issue below is intentional — the full catalog is in `datasets/DQ-EDGE-CASES.md`.

Docs: `datasets/README.md` (Phases 2 & 3)

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv('datasets/raw/customers.csv')
products = pd.read_csv('datasets/raw/products.csv')
orders = pd.read_csv('datasets/raw/orders.csv', parse_dates=['order_date'])
reviews = pd.read_json('datasets/raw/reviews.json')

### 1. Missing values

In [ ]:
# CSV already reads empty fields as NaN
customers.isnull().sum()

In [ ]:
# Where are the missing emails?
customers[customers['email'].isna()][['customer_id', 'first_name', 'email']].head()

### 2. Duplicates — same email, different ID

In [ ]:
customers[customers.duplicated('email', keep=False)].sort_values('email').head(20)

In [ ]:
# How many emails appear more than once?
dup_emails = customers['email'].dropna().value_counts()
dup_emails[dup_emails > 1].count()

### 3. Type conversion — parse dates

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['signup_date'].dtype

### 4. Invalid values — future signup dates

In [ ]:
future = customers[customers['signup_date'] > pd.Timestamp.now()]
future[['customer_id', 'signup_date']]

### 5. String standardization — states

In [ ]:
customers['state'].unique()

In [ ]:
# Map full names back to 2-letter codes
state_map = {'California': 'CA', 'New York': 'NY', 'Texas': 'TX', 'Florida': 'FL'}
customers['state'] = customers['state'].replace(state_map)
customers['state'].unique()

### 6. Normalize product categories

In [ ]:
products['category'].unique()

In [ ]:
products['category_clean'] = products['category'].str.replace('_', ' ').str.title()
products['category_clean'].unique()

### 7. Out-of-range values — ratings should be 1–5

In [ ]:
reviews['rating'].value_counts().sort_index()

In [ ]:
reviews[~reviews['rating'].between(1, 5)][['review_id', 'product_id', 'rating']]

### 8. A validation function

In [ ]:
def validate_orders(df):
    issues = []
    if (df['total'] < 0).any():
        issues.append('negative totals')
    if df['order_date'].max() > pd.Timestamp.now():
        issues.append('future order dates')
    return issues

validate_orders(orders)